In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import cuda, float64, complex128
from numba.cuda import jit as cuda_jit
import math

import few

from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode import KerrEccEqFlux
from few.amplitude.ampinterp2d import AmpInterpKerrEccEq
from few.summation.interpolatedmodesum import InterpolatedModeSum 


from few.utils.ylm import GetYlms

from few import get_file_manager

from few.waveform import FastKerrEccentricEquatorialFlux

from few.utils.geodesic import get_fundamental_frequencies

from few.utils.constants import YRSID_SI
from few.waveform import GenerateEMRIWaveform, FastSchwarzschildEccentricFlux, FastKerrEccentricEquatorialFlux


import os
import sys

# Change to the desired directory
os.chdir('/nfs/home/svu/e1498138/localgit/FEWNEW/work/')

# Add it to Python path
sys.path.insert(0, '/nfs/home/svu/e1498138/localgit/FEWNEW/work/')

import GWfuncs
import loglike
import modeselector
import dynesty
# import gc
# import pickle
import cupy as cp

# tune few configuration
cfg_set = few.get_config_setter(reset=True)
cfg_set.set_log_level("info")

import stableemrifisher
stableemrifisher.__file__
from tqdm import tqdm
from stableemrifisher.fisher.fisher import StableEMRIFisher


startup


In [4]:
# GPU configuration
use_gpu = True
dt = 10     # Time step
T = 0.25   # Total time

# keyword arguments for inspiral generator 
inspiral_kwargs={
        "func": 'KerrEccEqFlux',
        "DENSE_STEPPING": 0, #change to 1/True for uniform sampling
        "include_minus_m": False, 
        "err": 1e-15  # Error tolerance 
}

# keyword arguments for inspiral generator 
amplitude_kwargs = {
    "force_backend": "cuda12x" # Force GPU
}

# keyword arguments for Ylm generator (GetYlms)
Ylm_kwargs = {
    "force_backend": "cuda12x",  # Force GPU
}

# keyword arguments for summation generator (InterpolatedModeSum)
sum_kwargs = {
    "force_backend": "cuda12x",  # Force GPU
    "pad_output": True
}

waveform_class = FastKerrEccentricEquatorialFlux
waveform_class_kwargs = dict(inspiral_kwargs=inspiral_kwargs,
                             amplitude_kwargs=amplitude_kwargs,
                             Ylm_kwargs=Ylm_kwargs,
                             sum_kwargs=sum_kwargs,
                             use_gpu=use_gpu)

 
#waveform generator setup
waveform_generator = GenerateEMRIWaveform
waveform_generator_kwargs = dict(frame='detector')

waveform_gen = GenerateEMRIWaveform(
    waveform_class, 
    frame='detector',
    inspiral_kwargs=inspiral_kwargs, 
    amplitude_kwargs=amplitude_kwargs, 
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs,
    use_gpu=use_gpu
)

gwf = GWfuncs.GravWaveAnalysis(T, dt)

from lisatools.sensitivity import get_sensitivity, CornishLISASens


# from stableemrifisher.noise import sensitivity_LWA

sef = StableEMRIFisher(waveform_class=waveform_class, 
                       waveform_class_kwargs=waveform_class_kwargs,
                       waveform_generator=waveform_generator,
                       waveform_generator_kwargs=waveform_generator_kwargs,
                      stats_for_nerds = True, use_gpu = use_gpu,
                      deriv_type='stable', noise_model=get_sensitivity, noise_kwargs={'sens_fn':CornishLISASens, 'return_type': 'PSD'}, channels=["A"])


In [6]:
wave_params = {
    "m1": 1e6,
    "m2": 3e1,
    "a": 0.7,
    "p0": 7.5,
    "e0": 0.4,
    "xI0": 1.0,
    "dist": 0.5,
    "qS": 0.5,
    "phiS": 1,
    "qK": 1,
    "phiK": 1+np.pi/3,
    "Phi_phi0": 0.4,
    "Phi_theta0": 0.0,
    "Phi_r0": 0.5,
}

param_names = [
     "m1",
     "m2",
     "a",
    "p0",
    "e0",
 ]

In [7]:
fisher_matrix = sef(
    wave_params,
    param_names=param_names,
    # deltas=deltas,
)


TypeError: StableEMRIFisher.__call__() missing 13 required positional arguments: 'm2', 'a', 'p0', 'e0', 'xI0', 'dist', 'qS', 'phiS', 'qK', 'phiK', 'Phi_phi0', 'Phi_theta0', and 'Phi_r0'